### RNN (Recurrent Neural Network):

Recurrent Neural Network is a type of a neural network that have memory as a power to keep the context it learned from previous time-steps. We know that the hidden layer learns the patterns with the help of its neurons and activation function. So does the hidden state in the RNN but the difference is it uses the learnt stuff in next time step and so on.

In each time-step, it takes two input:

- the h(t-1) i.e. previous hidden state(context/memory/sentiment/meaning)it learned so far
- the current input x(t) in the respective time step

where the hidden state gives importance to each input, wh for previous hidden state and wi for the input in that time step.

It is always same for the whole dataset/batch, and after each backpropagation, it is updated.

At timestep t,
h(t) = (wh * h(t-1) + wi * x(t))

The main idea of the RNN is to consume the sequential data in such a way that it uses the context/meaning in the next timestep, keep summerizes the pattern from the start to the end to produce the required answer.

As the sequential data are the type of the data where the order matters, which means that the order in which the data exist gives meaning and will change/deflect entire meaning if order disrupts.

For example, for predicting the stock price in d5 (day5):
we need data in the order as d1 - d2 - d3 - d4 to summarize the pattern so that we can find the way the data is moving to make correct prediction.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import re

since the data is a text or natural language so we perform the natural language process for the text dataset.

- We first import the data as usual with the read_csv method of pandas.
- Then we have to convert the text to the tokens
- make a voccabulary dictionary for the tokens with their corresponding numerical value which represents their index
- use the index_format of the question or the text
- we include the question as well as the asnwer in the same voccabulary
- when we get the text from the user then the first thing we will do is, tokenize the text, convert the text to number based on the voccabulary and represent by a common index if the token is no where in the voccab.
- pass it to the embedding to calculate the embed and then the embedding is given to the rnn with certain number of hidden nuerons in the hidden layer that are connected in recurrent way. and then for the output we use the linear layer that is connected to each neuron of hidden layer to the linear layer
- Here, the embedding we talk is of the nn.Embedding() that is a trainable embedding and the other embedding model like the word embedding etc are the trained embedding

In [2]:
df = pd.read_csv("100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


Since the RNN doesnot understand the test/question and the asnwer as it is in words so we need to ocnvert the words into meaning to represent the word in number/ vector

In [3]:
# these are the labels of the data
df.iloc[:, 1].unique().tolist()

['Paris',
 'Berlin',
 'Harper-Lee',
 'Jupiter',
 '100',
 'Leonardo-da-Vinci',
 '8',
 'Au',
 '1945',
 'Nile',
 'Tokyo',
 'Albert-Einstein',
 '32',
 'Mars',
 'George-Orwell',
 'Pound',
 'Delhi',
 'Newton',
 '7',
 'CO2',
 '2',
 'Alexander-Graham-Bell',
 'Canberra',
 'Pacific-Ocean',
 '299,792,458m/s',
 'Portuguese',
 'Alexander-Fleming',
 'Ottawa',
 'Whale',
 'Hydrogen',
 'Everest',
 'NewYork',
 'vangogh',
 'H2O',
 'Rome',
 'Japan',
 'Armstrong',
 'Avocado',
 '6',
 'Yuan',
 'Jane-Austen',
 'Fe',
 'Diamond',
 'Asia',
 'George-Washington',
 'Parrot',
 'Simpsons',
 'VaticanCity',
 'Saturn',
 'Shakespeare',
 'Nitrogen',
 '206',
 'Mercury',
 'Moscow',
 'Benjamin-Franklin',
 'Canada',
 'Yellow',
 'February',
 'Biology',
 'China',
 'Nectar',
 'Night',
 'Seoul',
 'Edison',
 'Oxygen',
 '12',
 'Egypt',
 'Octopus',
 'Christmas',
 'Yen',
 'Basketball',
 'Australia',
 'MargaretThatcher',
 'Cheetah',
 'Madrid',
 'CharlesBabbage',
 'MexicoCity',
 'Piano',
 'ChristopherColumbus',
 'Pinocchio',
 'JamesCam

In [4]:
df.shape

(90, 2)

In [5]:
# tokenization of the text to tokens(smallest part of the text):
def tokenizer(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', "", text)
    return text.split()

In [6]:
# voccabulary
voccab = {
    '<UNK>' : 0
    }

In [7]:
# converts the words/tokens to voccabulary
def build_voccab(rows):
    for row in rows:
        tokenized_quetion = tokenizer(row[0])
        tokenized_answer = tokenizer(row[1])
        mereged_tokens = tokenized_quetion + tokenized_answer
        for token in mereged_tokens:
            if token not in voccab:
                voccab[token] = len(voccab)
    print("done ✅")

In [8]:
build_voccab(df.iloc[:, :].to_numpy())

done ✅


In [9]:
def text_to_indices(text, voccab):
    indexed_text = []
    tokens_in_text = tokenizer(text)
    for token in tokens_in_text:
        if token in voccab:
            indexed_text.append(voccab[token])
        else:
            indexed_text.append(voccab['<UNK>'])
    return indexed_text

In [10]:
text = "what is the capital of France?"
text_to_indices(text, voccab)

[1, 2, 3, 4, 5, 6]

In [11]:
# main data required for the rnn training:
voccab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harperlee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardodavinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'alberteinstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'georgeorwell': 68,
 'currency': 69,
 'united': 

In [12]:
class CustomDataset(Dataset):

    def __init__(self, df, voccab):
        self.df = df
        self.voccab = voccab

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        indexed_question = text_to_indices(self.df.iloc[index, 0], self.voccab)
        indexed_answer = text_to_indices(self.df.iloc[index, 1], self.voccab)
        return torch.tensor(indexed_question), torch.tensor(indexed_answer)

In [13]:
dataset = CustomDataset(df, voccab)

In [14]:
dataset.__len__()

90

In [15]:
dataset.__getitem__(89)

(tensor([ 42, 137,   2,  62,  39,   3, 322, 323]), tensor([6]))

In [19]:
dataset[89]

(tensor([ 42, 137,   2,  62,  39,   3, 322, 323]), tensor([6]))

In [213]:
dataloader = DataLoader(dataset = dataset, batch_size=1, shuffle= True)

In [481]:
class SentimentClassifierRNN(nn.Module):
    def __init__(self, voccab_size):
        super().__init__()
        self.embedding = nn.Embedding(voccab_size, embedding_dim=50)
        self.rnn = nn.RNN(50, 64)
        self.linear = nn.Linear(64, voccab_size)
    
    def forward(self, question):
        embed = self.embedding(question)
        rnn = self.rnn(embed)
        self.output = self.linear(rnn[1])
        return self.output


In [482]:
model = SentimentClassifierRNN(len(voccab))

In [483]:
lr = 0.001
epochs = 25

In [484]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [485]:
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader:
        question = question.reshape(len(question[0]))
        answer = answer.reshape(len(answer))
        optimizer.zero_grad()
        output = model.forward(question)
        loss = loss_function(output, answer)
        total_loss = total_loss + loss.item()
        loss.backward()
        optimizer.step()
    print("epoch:", epoch + 1, "loss:", total_loss/len(dataloader))

epoch: 1 loss: 5.813434574339125
epoch: 2 loss: 4.948538706037733
epoch: 3 loss: 4.095709371566772
epoch: 4 loss: 3.4864746040768093
epoch: 5 loss: 2.9337794171439278
epoch: 6 loss: 2.410834515094757
epoch: 7 loss: 1.9293202612135145
epoch: 8 loss: 1.508887912829717
epoch: 9 loss: 1.1661502858002981
epoch: 10 loss: 0.8893749594688416
epoch: 11 loss: 0.6859637028641171
epoch: 12 loss: 0.5354539205630621
epoch: 13 loss: 0.42780518184105554
epoch: 14 loss: 0.3485240941246351
epoch: 15 loss: 0.28725132677290177
epoch: 16 loss: 0.23801284838053915
epoch: 17 loss: 0.20160784588919745
epoch: 18 loss: 0.17361175823542807
epoch: 19 loss: 0.14910007036394543
epoch: 20 loss: 0.12898393885956871
epoch: 21 loss: 0.11276599483357536
epoch: 22 loss: 0.09937313654356533
epoch: 23 loss: 0.08800404858258036
epoch: 24 loss: 0.07842629407015111
epoch: 25 loss: 0.06969444366792837


In [533]:
index = text_to_indices("what is my name", voccab)

In [534]:
index

[1, 2, 0, 0]

In [535]:
output = model.forward(torch.tensor(index))

In [536]:
output.shape

torch.Size([1, 324])

In [537]:
labels = output.reshape(output.shape[1])

In [538]:
labels

tensor([ 1.1843,  1.0476,  1.0131,  1.1727,  1.0168,  1.2806,  0.2072, -1.2497,
         0.8697,  0.1551,  0.8569,  0.9520,  1.1246,  0.9410,  0.4356,  1.0266,
         0.3611,  1.1068,  0.9840,  0.8375,  1.1312,  0.9677,  1.7497, -0.1411,
         0.8379,  0.7457,  0.6066,  1.3001, -0.2680,  1.4653,  1.3636,  0.6230,
        -0.8215,  1.1117,  1.4897,  0.5509, -0.2212,  0.7766,  0.8222,  1.2327,
         0.9584, -1.3205,  1.0059,  0.9300,  1.2038,  0.9965,  1.3724,  1.3453,
         0.9259, -1.6215,  1.2228,  0.9762, -1.8046,  1.2180,  1.5415,  0.9773,
         1.4284,  1.4392, -1.8435,  1.5346,  0.9314,  0.6790,  1.2340,  0.8307,
         0.9790,  0.1035,  0.7037,  0.6339,  1.2207,  0.4167,  1.4541,  0.9415,
        -0.7618,  1.1352,  1.7748,  1.1055,  0.8851, -0.5005,  0.7639,  1.2992,
         1.2995,  0.8399,  1.5545,  0.7030,  1.2959,  0.6945,  0.8333,  1.4551,
         1.1362,  0.8651,  0.5113, -0.5304,  1.1100,  0.9766,  1.6061,  1.6159,
         1.0452,  1.3483, -0.6972,  0.20

In [539]:
index = torch.max(labels, dim = 0)[1]

In [549]:
key = [k for k, v in voccab.items() if v == index]

In [553]:
"".join(key)

'yen'